# GlobalCLIP -- 02b: Train QLayer Interference Model

Trains `GlobalCLIPQLayerModel`: a frozen PARNET backbone + `MixCoeffHead`
+ `QLayer` (quantum interference) + dilated CNN refinement.

**Architecture:**
```
RNA sequence (4×600)
    │
    ▼
PARNET backbone  [frozen]
    │              │
    ▼              ▼
(B,512,L)      (B,223,L) RBP log-prob tracks
    │              │
    ▼              │ × exp(log_scale)
MixCoeffHead → alpha (B,223)   [sigmoid]
                   │
                   ▼
    QLayer  ψ_i = A_i · e^{iφ_i}      (223 learnable phases)
    I(p) = |Σ_i α_i · ψ_i(p)|²  →  (B,1,L)
                   │
          Dilated CNN refinement
                   │
                   ▼
            (B,1,L) GlobalCLIP prediction
```

The QLayer cross-terms `2·α_i·α_j·A_i·A_j·cos(φ_i−φ_j)` capture protein-protein
interactions with only 223 phase parameters.  After training on log-FE signal,
background-noise proteins naturally converge to φ ≈ φ_signal + π (destructive
interference → self-cancelling).

**Training target:** log(1+signal) − log(1+control)

**Loss:** Pearson + Multinomial NLL + alpha sparsity + phase L2 regularisation


## Set-up

### Imports

In [1]:
import pylbsr.notebooks
import pylbsr.misc

import torch
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightning.pytorch as pl
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from pathlib import Path
from dotmap import DotMap

from parnet_additional_utils import (
    load_parnet_model,
    ParnetModelName,
)
from globalclip_utils import (
    GlobalCLIPQLayerModel,
    GlobalCLIPLightningModule,
    GlobalCLIPDataset,
    save_run_config,
)


/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/gin/config.py:615: FutureWarning: `NLLLoss2d` has been deprecated. Please use `NLLLoss` instead as a drop-in replacement and see https://pytorch.org/docs/main/nn.html#torch.nn.NLLLoss for more details.
  decorated_class = decorating_meta(cls.__name__, (cls,), overrides)
Seed set to 42


### Initialisation

In [2]:
_notebook_name = "02b_train_qlayer.py.ipynb"
_notebook_path = f"notebooks/globalclip/{_notebook_name}"

pylbsr.notebooks.enable_cell_timing_metadata(show=True)
logger = pylbsr.misc.init_logger(_notebook_name)
PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)
logger.info(f"Project directory: {PROJECT_DIR}")


[10:00:07] INFO - Project directory: /mnt/storage1/workspace/pgoldemund/parnet--idea1-reconstruction-head


### Parameters

In [3]:
params_gpu_index          = 0
params_run_id             = "globalclip.qlayer.v2"

# Dataset
params_control_dataset    = "globalclip_lysate_noNHS"   # cleanest background
params_seq_length         = 600
params_batch_size         = 64
params_num_rbps           = 223

# Model -- MixCoeffHead
params_mix_hidden         = 128

# Model -- CNN after QLayer
params_cnn_channels       = 64
params_cnn_kernel         = 9
params_cnn_layers         = 3    # uses dilation 1, 2, 4

# Training
params_lr                 = 1e-4
params_max_epochs         = 50
params_lambda_nll         = 0.3
params_lambda_alpha       = 5.0
params_lambda_phase       = 0.01   # L2 regularisation on QLayer phases
params_early_stop_patience= 8
params_num_workers        = 4


⏱ 0.00 s (00:00:00)


### Filepaths and device

In [4]:
pylbsr.misc.set_seed(42)

device = torch.device(f"cuda:{params_gpu_index}" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(params_gpu_index)
    logger.info(f"GPU: {torch.cuda.get_device_name(device)}")
else:
    logger.warning("No GPU available — running on CPU (will be slow).")

_fp_cfg = yaml.safe_load((PROJECT_DIR / "config" / "filepaths.server.yaml").read_text())
pretrained_model_name = ParnetModelName.PARNET_7M_0_0

def _res(p):
    p = Path(p)
    return p if p.is_absolute() else PROJECT_DIR / p

FILEPATHS = DotMap()
FILEPATHS.pretrained_model = _res(_fp_cfg["models"][pretrained_model_name.value])
FILEPATHS.dataset          = _res(_fp_cfg["data"][params_control_dataset]["pt"])
FILEPATHS.rbp_names        = PROJECT_DIR / "results" / "globalclip" / "datasets" / "rbp_names.txt"
FILEPATHS.output_dir       = PROJECT_DIR / _fp_cfg["results"]["qlayer_model"] / params_run_id
FILEPATHS.output_dir.mkdir(parents=True, exist_ok=True)

for k, v in FILEPATHS.items():
    logger.info(f"{k:25s}: {v}")


[10:00:08] INFO - GPU: NVIDIA RTX A4000
[10:00:08] INFO - pretrained_model         : /mnt/storage1/ml4rg26-shared/parnet-eclip/models-full-rbp-set/parnet.7m-0.0.pt
[10:00:08] INFO - dataset                  : /mnt/storage1/ml4rg26-deconvgclip/provided_data/600nt_globalCLIP_synchronized_datasets/globalclip_lysate_noNHS_600bp_signalfiltered.pt.gz
[10:00:08] INFO - rbp_names                : /mnt/storage1/workspace/pgoldemund/parnet--idea1-reconstruction-head/results/globalclip/datasets/rbp_names.txt
[10:00:08] INFO - output_dir               : /mnt/storage1/workspace/pgoldemund/parnet--idea1-reconstruction-head/results/globalclip/qlayer/globalclip.qlayer.v1


Seed set to 42
⏱ 0.11 s (00:00:00)


## Load data

In [5]:
train_ds = GlobalCLIPDataset(FILEPATHS.dataset, split="train",
                              seq_len=params_seq_length, total_key="globalCLIP")
val_ds   = GlobalCLIPDataset(FILEPATHS.dataset, split="valid",
                              seq_len=params_seq_length, total_key="globalCLIP")

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=params_batch_size, shuffle=True,
    num_workers=params_num_workers, pin_memory=torch.cuda.is_available()
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=params_batch_size, shuffle=False,
    num_workers=params_num_workers, pin_memory=torch.cuda.is_available()
)

logger.info(f"Train: {len(train_ds)} samples  Val: {len(val_ds)} samples")

batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"  batch['{k}']: {tuple(v.shape)}")


Loading (gz) globalclip_lysate_noNHS_600bp_signalfiltered.pt.gz split='train'... loaded 39052 samples.
Loading (gz) globalclip_lysate_noNHS_600bp_signalfiltered.pt.gz split='valid'... loaded 7361 samples.


[10:00:31] INFO - Train: 39052 samples  Val: 7361 samples


  batch['sequence']: (64, 4, 600)
  batch['signal']: (64, 1, 600)
⏱ 23.83 s (00:00:23)


## Build QLayer model

In [6]:
logger.info(f"Loading pretrained PARNET from {FILEPATHS.pretrained_model}")
parnet = load_parnet_model(
    pretrained_model_name,
    FILEPATHS.pretrained_model,
    dtype=torch.float32,
    device=device,
)
parnet.eval()
logger.info("PARNET loaded (223-task head kept frozen).")


[10:00:31] INFO - Loading pretrained PARNET from /mnt/storage1/ml4rg26-shared/parnet-eclip/models-full-rbp-set/parnet.7m-0.0.pt
[10:00:32] INFO - PARNET loaded (223-task head kept frozen).


⏱ 0.20 s (00:00:00)


In [7]:
model = GlobalCLIPQLayerModel(
    parnet_model=parnet,
    num_rbps=params_num_rbps,
    mix_hidden=params_mix_hidden,
    cnn_channels=params_cnn_channels,
    cnn_kernel=params_cnn_kernel,
    cnn_layers=params_cnn_layers,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
logger.info(f"Parameters: {trainable:,} trainable / {total:,} total")
logger.info(f"  -- MixCoeffHead: {sum(p.numel() for p in model.mix_coeff.parameters()):,}")
logger.info(f"  -- QLayer phases: {model.qlayer.phase.numel()} (one per RBP)")
logger.info(f"  -- CNN: {sum(p.numel() for p in model.cnn.parameters()):,}")

# Sanity check
with torch.no_grad():
    pred, alpha = model(batch["sequence"].to(device))
print(f"pred shape : {tuple(pred.shape)}")
print(f"alpha shape: {tuple(alpha.shape)}")
print(f"Initial phases (first 5): {model.qlayer.phase[:5].detach().cpu().numpy()}")


[10:00:32] INFO - Parameters: 169,438 trainable / 7,704,253 total
[10:00:32] INFO -   -- MixCoeffHead: 94,431
[10:00:32] INFO -   -- QLayer phases: 223 (one per RBP)
[10:00:32] INFO -   -- CNN: 74,561
/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(


pred shape : (64, 1, 600)
alpha shape: (64, 223)
Initial phases (first 5): [0. 0. 0. 0. 0.]
⏱ 0.41 s (00:00:00)


## Train

In [8]:
lightning_model = GlobalCLIPLightningModule(
    model=model,
    lr=params_lr,
    lambda_nll=params_lambda_nll,
    lambda_alpha=params_lambda_alpha,
    lambda_phase=params_lambda_phase,
)

callbacks = [
    ModelCheckpoint(
        dirpath=FILEPATHS.output_dir / "checkpoints",
        filename="best-{epoch:02d}-{val/loss:.4f}",
        monitor="val/loss",
        mode="min",
        save_top_k=2,
    ),
    EarlyStopping(
        monitor="val/loss",
        patience=params_early_stop_patience,
        mode="min",
    ),
]

loggers = [
    CSVLogger(str(FILEPATHS.output_dir), name="csv_logs"),
    TensorBoardLogger(str(FILEPATHS.output_dir), name="tb_logs"),
]

trainer = pl.Trainer(
    max_epochs=params_max_epochs,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=[params_gpu_index] if torch.cuda.is_available() else 1,
    callbacks=callbacks,
    logger=loggers,
    log_every_n_steps=50,
    deterministic=True,
)

logger.info("Starting training...")
trainer.fit(lightning_model, train_loader, val_loader)
logger.info("Training complete.")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
[10:00:32] INFO - Starting training...
2026-07-01 10:00:32.835424: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-01 10:00:32.849470: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-01 10:00:32.870502: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-01 10:00:32.870554: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for pl

Sanity Checking: |                                                                                            …

/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:527: Found 62 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |                                                                                                   …


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

⏱ 25.35 s (00:00:25)


/home/pgoldemund/pixi-envs/parnet--unified-3311472763523369453/envs/parnet-dev-cu12/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Save model and run config

In [ ]:
torch.save(model.state_dict(), FILEPATHS.output_dir / "model.statedict.pt")
torch.save(model, FILEPATHS.output_dir / "model.full.pt")

run_cfg = {
    "model_type":            "GlobalCLIPQLayerModel",
    "pretrained_model_name":  pretrained_model_name.value,
    "control_dataset":        params_control_dataset,
    "params_seq_length":      params_seq_length,
    "params_batch_size":      params_batch_size,
    "params_num_rbps":        params_num_rbps,
    "params_mix_hidden":      params_mix_hidden,
    "params_cnn_channels":    params_cnn_channels,
    "params_cnn_kernel":      params_cnn_kernel,
    "params_cnn_layers":      params_cnn_layers,
    "params_lr":              params_lr,
    "params_max_epochs":      params_max_epochs,
    "params_lambda_nll":      params_lambda_nll,
    "params_lambda_alpha":    params_lambda_alpha,
    "params_lambda_phase":    params_lambda_phase,
    "dataset_path":           str(FILEPATHS.dataset),
    "output_dir":             str(FILEPATHS.output_dir),
}
save_run_config(FILEPATHS.output_dir, run_cfg)
logger.info(f"Saved to {FILEPATHS.output_dir}")


## Quick phase inspection

In [ ]:
rbp_names = (FILEPATHS.rbp_names).read_text().strip().split("\n") if FILEPATHS.rbp_names.exists() else [f"RBP_{i}" for i in range(223)]

phases = model.qlayer.phase.detach().cpu().numpy()
coupling = model.get_coupling_matrix().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of phases
axes[0].hist(phases, bins=30, color="steelblue", edgecolor="none", alpha=0.8)
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("Phase φ_i (radians)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of learned QLayer phases\n"
                  "φ≈0: constructive;  φ≈±π: destructive (noise suppression)")

# Polar plot of phases
ax_pol = fig.add_axes([0.55, 0.1, 0.4, 0.8], polar=True)
ax_pol.scatter(phases, np.ones_like(phases), alpha=0.5, s=20, color="steelblue")
ax_pol.set_rticks([])
ax_pol.set_title("Polar: protein phases", pad=15)

plt.savefig(FILEPATHS.output_dir / "phase_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Phase range: [{phases.min():.3f}, {phases.max():.3f}] rad")
print(f"Std dev of phases: {phases.std():.3f}")


## Training curves

In [ ]:
csv_log_dir = FILEPATHS.output_dir / "csv_logs"
metrics_paths = sorted(csv_log_dir.glob("version_*/metrics.csv"))
if metrics_paths:
    df_csv = pd.read_csv(metrics_paths[-1])
    epoch_df = df_csv.groupby("epoch").last().reset_index()

    plots = [
        ({"train": "train/loss_epoch", "val": "val/loss"}, "Total loss"),
        ({"train": "train/pearson_epoch", "val": "val/pearson"}, "Pearson loss"),
        ({"val phase_reg": "val/phase_reg"}, "Phase L2 regularisation"),
    ]
    fig, axes = plt.subplots(1, len(plots), figsize=(14, 4))
    for (col_dict, title), ax in zip(plots, axes):
        for label, col in col_dict.items():
            if col in epoch_df.columns:
                epoch_df.plot("epoch", col, ax=ax, label=label, marker="o", markersize=3)
        ax.set_title(title)
        ax.legend(fontsize=8)
    plt.suptitle(f"Training metrics -- {params_run_id}")
    plt.tight_layout()
    plt.savefig(FILEPATHS.output_dir / "training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()


## Next steps

- Run `03_evaluate_and_analyze.py.ipynb` to compare both models on the test set
- Inspect the coupling matrix `model.get_coupling_matrix()` to see which proteins cooperate
- The polar plot of phases reveals cooperative clusters (same φ) vs noise suppressors (φ+π)
